In [1]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional
import cftime

import glob
import numpy as np
import xarray as xr
import xcdat as xc
import pandas as pd

# Optional extras (only used if enabled)
import scipy.fftpack
from scipy.signal import detrend as scipy_detrend
import pywt

from pathlib import Path

from ExperimentUtil import (
    run_dict,
    open_var_from_files,
    get_comp_dir,
    find_nc_files,
    collect_variable_list,
    flatten_variable_dict,
    open_compvar,
    set_monthly_time_to_midmonth_cftime
)

In [24]:
@dataclass(frozen=True)
class QboConfig:
    u_var: str = "U"
    time_name: str = "time"
    p_name: str = "plev"
    lat_name: str = "lat"
    lon_name: str = "lon"
    lat_band: tuple[float, float] = (-5.0, 5.0)

    pow_spec_lev_hpa: float = 20.0
    p_units_target: str = "hPa"

    amp_remove_seasonal_cycle: bool = False

    # ADD THIS
    section_name: str = "qbo"
    var: str = u_var
    
@dataclass
class QboMetrics:
    name: str
    qbo: xr.DataArray                 # (time, p)
    amp_p2p: xr.DataArray             # (p,)
    amp_rms: xr.DataArray             # (p,)
    amp_rms_detrend: xr.DataArray     # (p,)

    # Optional (may be NaN placeholders)
    amp_20_40_fft: xr.DataArray       # (p,)
    psd_period: xr.DataArray          # (period,)
    psd_amp: xr.DataArray             # (period,)
    wave_period: xr.DataArray         # (period,)
    wavelet: xr.DataArray             # (period,)

    def to_dataset(self) -> xr.Dataset:
        ds = xr.Dataset()
        ds["qbo"] = self.qbo
        ds["amp_p2p"] = self.amp_p2p
        ds["amp_rms"] = self.amp_rms
        ds["amp_rms_detrend"] = self.amp_rms_detrend
        ds["amp_20_40_fft"] = self.amp_20_40_fft
        ds["psd_period"] = self.psd_period
        ds["psd_amp"] = self.psd_amp
        ds["wave_period"] = self.wave_period
        ds["wavelet"] = self.wavelet
        ds.attrs["name"] = self.name
        return ds


class QboAnalyzerFromScratch:
    """
    QBO-like analyzer for monthly U winds (time, plev), optionally with lat/lon.

    - Produces defensible short-record metrics by default (no FFT/PSD/wavelet).
    - Can optionally compute spectral diagnostics if record length is sufficient.
    """

    def __init__(self, cfg: QboConfig):
        self.cfg = cfg

    # -------------------------
    # Core preprocessing
    # -------------------------
    def _unify_plev(self, da: xr.DataArray) -> xr.DataArray:
        z_axis = xc.get_dim_coords(da, axis="Z")
        z_dim = z_axis.name

        out = da.copy()

        # Convert Pa -> hPa if needed (robust units check)
        units = (z_axis.attrs.get("units", "") or "").strip().lower()
        if units == "pa" and self.cfg.p_units_target.strip().lower() == "hpa":
            out = out.assign_coords({z_dim: z_axis / 100.0})
            out[z_dim].attrs["units"] = "hPa"

        # Sort pressure coordinate for consistent downstream behavior
        out = out.sortby(z_dim, ascending=True)
        return out

    def _extract_qbo_time_height(self, data: xr.Dataset | xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
    
        if isinstance(data, xr.Dataset):
            if cfg.var not in data:
                raise KeyError(f"'{cfg.var}' not found in dataset.")
            u = data[cfg.var]
        else:
            u = data
    
        if cfg.time_name in u.dims:
            u = u.sortby(cfg.time_name)
    
        if cfg.lat_name in u.dims:
            lat0, lat1 = cfg.lat_band
            lat = u[cfg.lat_name]
            if lat[0] < lat[-1]:
                u = u.sel({cfg.lat_name: slice(lat0, lat1)})
            else:
                u = u.sel({cfg.lat_name: slice(lat1, lat0)})
            u = u.mean(cfg.lat_name, keep_attrs=True)
    
        if cfg.lon_name in u.dims:
            u = u.mean(cfg.lon_name, keep_attrs=True)
    
        u = self._unify_plev(u)
    
        u = u.rename(cfg.section_name)
        u.attrs["description"] = f"Equatorial (5S–5N) zonal-mean {cfg.var} time-height section"
        return u
    
    # -------------------------
    # Defensible amplitude metrics
    # -------------------------
    def _amp_p2p(self, qbo: xr.DataArray) -> xr.DataArray:
        t = self.cfg.time_name
        amp = 0.5 * (qbo.max(t) - qbo.min(t))
        amp = amp.rename("amp_p2p")
        amp.attrs["long_name"] = "Half peak-to-peak amplitude over analysis window"
        return amp

    def _amp_rms(self, qbo: xr.DataArray, detrend: bool = False) -> xr.DataArray:
        t = self.cfg.time_name
        x = qbo

        if detrend:
            # detrend each level along time
            vals = x.transpose(t, ...).values  # (time, p)
            vals_dt = np.apply_along_axis(scipy_detrend, 0, vals)
            x = xr.DataArray(vals_dt, coords=x.coords, dims=x.dims, name=x.name)

        anom = x - x.mean(t)
        amp = np.sqrt((anom ** 2).mean(t))
        amp = amp.rename("amp_rms_detrend" if detrend else "amp_rms")
        amp.attrs["long_name"] = "RMS amplitude of anomalies over analysis window"
        amp.attrs["detrended"] = str(detrend)
        return amp

    # -------------------------
    # Optional spectral metrics (NOT recommended for 2 years)
    # -------------------------
    @staticmethod
    def _deseason(xraw: np.ndarray) -> np.ndarray:
        """
        Remove the month-of-year climatology.
        Works for partial years and avoids trailing zero-fill.
        """
        months = np.arange(len(xraw)) % 12
        clim = np.array([np.nanmean(xraw[months == m]) for m in range(12)])
        return xraw - clim[months]

    def _amp_20_40_fft(self, qbo: xr.DataArray, min_months_for_spectral: int = 96) -> xr.DataArray:
        """
        Computes 20–40 month band amplitude per level via FFT (requires long record).
        Returns NaNs if too short.
        """
        t = self.cfg.time_name
        z = xc.get_dim_coords(qbo, axis="Z")
        z_dim = z.name

        n_months = qbo.sizes[t]
        out = xr.DataArray(
            np.full((qbo.sizes[z_dim],), np.nan),
            coords={z_dim: qbo[z_dim]},
            dims=(z_dim,),
            name="amp_20_40_fft",
        )
        out.attrs["warning"] = "Not meaningful for short records (e.g., 24 months)."

        if n_months < min_months_for_spectral:
            return out

        q = np.squeeze(qbo.values)  # time x lev
        amp = np.zeros(qbo.sizes[z_dim], dtype=float)

        for ilev in range(qbo.sizes[z_dim]):
            y_input = self._deseason(np.squeeze(q[:, ilev]))
            y = scipy.fftpack.fft(y_input)
            n = len(y)

            frequency = np.arange(n / 2) / n  # cycles/month
            period = 1 / frequency
            values = y[0:int(np.floor(n / 2))]
            fyy = values * np.conj(values)

            psd_sum = 2 * np.nansum(fyy[(period <= 40) & (period >= 20)])
            amp[ilev] = np.sqrt(2 * psd_sum) * (frequency[1] - frequency[0])

        out.values = amp
        out.attrs["long_name"] = "FFT-derived 20–40 month band amplitude (requires long record)"
        return out

    def _psd_at_level(self, qbo: xr.DataArray, min_months_for_spectral: int = 96) -> tuple[xr.DataArray, xr.DataArray]:
        """
        PSD amplitude spectrum at nearest 20 hPa (requires long record).
        Returns NaNs if too short.
        """
        t = self.cfg.time_name
        n_months = qbo.sizes[t]

        period = xr.DataArray(np.array([np.nan]), dims=("period",), name="psd_period")
        amp = xr.DataArray(np.array([np.nan]), dims=("period",), name="psd_amp")
        if n_months < min_months_for_spectral:
            return period, amp

        z = xc.get_dim_coords(qbo, axis="Z")
        z_list = list(z.values)
        closest = min(z_list, key=lambda x: abs(x - self.cfg.pow_spec_lev_hpa))
        idx = z_list.index(closest)

        xraw = qbo.values[:, idx]
        x_deseasoned = self._deseason(xraw)

        # FFT settings
        L0 = len(xraw)
        NFFT0 = int(2 ** np.ceil(np.log2(L0)))
        x0 = scipy.fftpack.fft(x_deseasoned, n=NFFT0) / L0
        freq = np.arange(0, (NFFT0 / 2 + 1)) / NFFT0  # cycles/month
        per = 1 / freq
        amp0 = 2 * np.abs(x0[0:int(NFFT0 / 2 + 1)])

        # Flip so period increases for plotting
        per_f = per[::-1]
        amp_f = amp0[::-1]

        # interpolate to common period axis
        period_new = np.concatenate((np.arange(2.0, 33.0), np.arange(34.0, 100.0, 2.0)))
        amp_new = np.interp(period_new, per_f[:-1], amp_f[:-1])

        return (
            xr.DataArray(period_new, dims=("period",), name="psd_period"),
            xr.DataArray(amp_new, dims=("period",), name="psd_amp"),
        )

    def _wavelet_at_level(self, qbo: xr.DataArray, min_months_for_spectral: int = 96) -> tuple[xr.DataArray, xr.DataArray]:
        """
        Wavelet spectrum at nearest 20 hPa (requires long record).
        Returns NaNs if too short.
        """
        t = self.cfg.time_name
        n_months = qbo.sizes[t]

        period = xr.DataArray(np.array([np.nan]), dims=("period",), name="wave_period")
        wave = xr.DataArray(np.array([np.nan]), dims=("period",), name="wavelet")
        if n_months < min_months_for_spectral:
            return period, wave

        z = xc.get_dim_coords(qbo, axis="Z")
        z_list = list(z.values)
        closest = min(z_list, key=lambda x: abs(x - self.cfg.pow_spec_lev_hpa))
        idx = z_list.index(closest)

        data = qbo.values[:, idx]
        data = data - np.nanmean(data)
        data = scipy_detrend(data)

        per = np.arange(1, 56)  # months
        widths = per
        cfs, _ = pywt.cwt(data, scales=widths, wavelet="cmor1.5-1.0")
        psd = np.mean(np.square(np.abs(cfs)), axis=1)
        psd = np.sqrt(psd)

        return (
            xr.DataArray(per, dims=("period",), name="wave_period"),
            xr.DataArray(psd, dims=("period",), name="wavelet"),
        )
        
    def _remove_monthly_clim(self, x: xr.DataArray) -> xr.DataArray:
        t = self.cfg.time_name
        g = x.groupby(f"{t}.month")     # robust with cftime/object times
        clim = g.mean(t)               # (month, plev)
        out = g - clim                 # back to (time, plev)
        # keep things clean (optional)
        if "month" in out.coords:
            out = out.reset_coords("month", drop=True)
        return out
    
    # -------------------------
    # Public API
    # -------------------------
    def compute_metrics(
        self,
        name: str,
        data: xr.Dataset | xr.DataArray,
        enable_spectral: bool = False,
        min_months_for_spectral: int = 96,
    ) -> QboMetrics:
        
        qbo = self._extract_qbo_time_height(data)
        
        qbo_for_amp = qbo
        if self.cfg.amp_remove_seasonal_cycle:
            qbo_for_amp = self._remove_monthly_clim(qbo)
            
        amp_p2p = self._amp_p2p(qbo_for_amp)
        amp_rms = self._amp_rms(qbo_for_amp, detrend=False)
        amp_rms_dt = self._amp_rms(qbo_for_amp, detrend=True)

        n_months = qbo.sizes[self.cfg.time_name]
        do_spec = enable_spectral and (n_months >= min_months_for_spectral)

        amp_20_40 = self._amp_20_40_fft(qbo, min_months_for_spectral=min_months_for_spectral) if do_spec else xr.zeros_like(amp_p2p) * np.nan
        psd_period, psd_amp = self._psd_at_level(qbo, min_months_for_spectral=min_months_for_spectral) if do_spec else (
            xr.DataArray(np.array([np.nan]), dims=("period",), name="psd_period"),
            xr.DataArray(np.array([np.nan]), dims=("period",), name="psd_amp"),
        )
        wave_period, wavelet = self._wavelet_at_level(qbo, min_months_for_spectral=min_months_for_spectral) if do_spec else (
            xr.DataArray(np.array([np.nan]), dims=("period",), name="wave_period"),
            xr.DataArray(np.array([np.nan]), dims=("period",), name="wavelet"),
        )
        
        # NEW: also apply to saved qbo itself (not just amplitude)
        #if self.cfg.amp_remove_seasonal_cycle:
        #    qbo = self._remove_monthly_clim(qbo)
            
        return QboMetrics(
            name=name,
            qbo=qbo,
            amp_p2p=amp_p2p,
            amp_rms=amp_rms,
            amp_rms_detrend=amp_rms_dt,
            amp_20_40_fft=amp_20_40,
            psd_period=psd_period,
            psd_amp=psd_amp,
            wave_period=wave_period,
            wavelet=wavelet,
        )

    def save_metrics(self, metrics: QboMetrics, out_nc: str | Path) -> Path:
        out_nc = Path(out_nc)
        out_nc.parent.mkdir(parents=True, exist_ok=True)
        metrics.to_dataset().to_netcdf(out_nc)
        return out_nc
    

In [25]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "qbo"
    out_dir.mkdir(parents=True, exist_ok=True)

    exp_list = ["CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE"]
    plev_name = 'plev'
    var_cfg = {
        "tag": "ta",
        "run_key": "cmip6",
        "sub": "post/atm/180x360_aave/cmip_ts",  # change if your real layout differs
        "name": "ta",
        "period": "2012-2013",
        "dims": ("time", "plev", "lat", "lon"),
        "lat_band": (-5.0, 5.0),
        "p_units_target": "hPa",
        "is_cmip6": True,
        "amp_remove_seasonal_cycle": True
    }
    
    vtag = var_cfg["tag"]
    run_key = var_cfg["run_key"]
    varname = var_cfg["name"]
    anom = var_cfg["amp_remove_seasonal_cycle"]
    lat_band = var_cfg["lat_band"]
    p_units_target = var_cfg["p_units_target"]
    time_name, p_name, lat_name, lon_name = var_cfg["dims"]
    
    analyzer = QboAnalyzerFromScratch(
        QboConfig(
            u_var=varname,              # "ta"
            time_name=time_name,
            p_name=p_name,
            lat_name=lat_name,
            lon_name=lon_name,
            lat_band=lat_band,
            p_units_target=p_units_target,
            amp_remove_seasonal_cycle=anom,   # <--- IMPORTANT for temperature
        )
    )
    
    for exp in exp_list:
        print(f"\n=== {exp} ({run_dict[exp]['name']}) ===")

        # Avoid mutating the global run_dict entry
        exp_run = dict(run_dict[exp])
        exp_run[run_key] = var_cfg["sub"]

        files = find_nc_files(
            exp_run,
            run_key,
            varname,
            is_cmip6=var_cfg["is_cmip6"],
        )
        if not files:
            raise FileNotFoundError(
                f"No files matched for {varname} under {run_key} -> {var_cfg['sub']} "
                f"(exp={exp}, period={var_cfg['period']})"
            )
            
        da = open_var_from_files(files, varname)
        
        # Rename lev -> plev if the data uses lev
        p_name_local = p_name  # keep analyzer's p_name untouched
        if "lev" in da.dims and "plev" not in da.dims:
            da = da.rename({"lev": "plev"})
        p_name_local = "plev" if "plev" in da.dims else p_name_local
        
        print("Updated time coordinate:")
        da = set_monthly_time_to_midmonth_cftime(da, time_name=time_name)  # infers DatetimeNoLeap automatically
        #print(da[time_name].values)
        # xarray will set CF calendar itself during to_netcdf; don't keep it in attrs
        da[time_name].attrs.pop("calendar", None)
        
        # Basic dimension checks
        need = {time_name, p_name_local}
        missing = need - set(da.dims)
        if missing:
            raise ValueError(f"{exp}: missing required dims {missing}. got dims={da.dims}")
        
        # Ensure pressure coord has units
        if p_name_local in da.coords:
            if not (da[p_name_local].attrs.get("units") or "").strip():
                pmax = float(np.nanmax(da[p_name_local].values))
                da[p_name_local].attrs["units"] = "Pa" if pmax > 2000 else "hPa"
        
        # If this exp ended up with lev (rare), rebuild analyzer for that exp
        analyzer_local = analyzer
        if analyzer.cfg.p_name != p_name_local:
            analyzer_local = QboAnalyzerFromScratch(
                QboConfig(
                    u_var=varname,
                    time_name=time_name,
                    p_name=p_name_local,
                    lat_name=lat_name,
                    lon_name=lon_name,
                    lat_band=lat_band,
                    p_units_target=p_units_target,
                    amp_remove_seasonal_cycle=anom,   # <-- ADD THIS
                )
            )
        
        metrics = analyzer_local.compute_metrics(
            name=f"{exp}_CMIP6_{varname}_{var_cfg['period']}",
            data=da,
            enable_spectral=False,
        )
        
        out_nc = out_dir / f"qbo_{exp}_{vtag}_{var_cfg['period']}.nc"
        analyzer_local.save_metrics(metrics, out_nc)
        print(f"Wrote {out_nc}")


=== CTRL (20260114.v3-LR-AMIP-CTRL1.ne30pg2_r05_IcoswISC30E3r5) ===
Updated time coordinate:


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/qbo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


Wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/qbo/qbo_CTRL_ta_2012-2013.nc

=== DARTOP (20260114.v3-LR-AMIP-DART1.ne30pg2_r05_IcoswISC30E3r5) ===
Updated time coordinate:


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/qbo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


Wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/qbo/qbo_DARTOP_ta_2012-2013.nc

=== ERA5OP (20260114.v3-LR-AMIP-ERA51.ne30pg2_r05_IcoswISC30E3r5) ===
Updated time coordinate:


...

Wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/qbo/qbo_DARTBL_ta_2012-2013.nc

=== DARTBA (20260114.v3-LR-AMIP-DARTBA.ne30pg2_r05_IcoswISC30E3r5) ===
Updated time coordinate:


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/qbo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


Wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/qbo/qbo_DARTBA_ta_2012-2013.nc

=== DARTAE (20260114.v3-LR-AMIP-DARTAE.ne30pg2_r05_IcoswISC30E3r5) ===
Updated time coordinate:


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/qbo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


Wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/qbo/qbo_DARTAE_ta_2012-2013.nc


In [26]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "qbo"
    out_dir.mkdir(parents=True, exist_ok=True)

    ref_dict = {
        "name": "ERA5",
        "path": "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/ERA5",
        "period": "2012-2013",
    }
    
    plev = [ 100000, 92500, 85000, 70000, 60000, 50000, 40000, 30000, 25000, 20000, 15000, 10000, 7000, 5000, 3000, 2000, 1000, 500, 100] 
    

    var_cfg = {
        "tag": "ta",
        "sub": "post/atm/180x360_aave/ts/monthly/1yr",
        "pattern": "T_*.nc",                 # matches U_201201_201212.nc, U_201301_201312.nc
        "name": "T",
        "dims": ("time", "lev", "lat", "lon"),
        "lat_band": (-5.0, 5.0),
        "p_units_target": "hPa",
        "amp_remove_seasonal_cycle": True
    }
    
    vtag = var_cfg["tag"]
    varname = var_cfg["name"]
    anom = var_cfg["amp_remove_seasonal_cycle"]
    lat_band = var_cfg["lat_band"]
    p_units_target = var_cfg["p_units_target"]
    time_name, p_name, lat_name, lon_name = var_cfg["dims"]
    
    base = Path(ref_dict["path"]) / var_cfg["sub"]
    files = sorted(glob.glob(str(base / var_cfg["pattern"])))
    if not files:
        raise FileNotFoundError(f"No ERA5 files matched: {base / var_cfg['pattern']}")

    print(f"\n=== {ref_dict['name']} ({ref_dict['period']}) ===")
    print(f"{vtag}: nfiles={len(files)} -> {[Path(f).name for f in files]}")

    da = open_var_from_files(files, varname)
    if p_name == "lev":
        da = da.rename({"lev": "plev"})
        p_name = "plev"
        
    # --- Simple, consistent monthly time: set all months to YYYY-MM-16 00:00 ---
    print("Updated time coordinate:")
    da = set_monthly_time_to_midmonth_cftime(da, time_name=time_name, calendar_cls=cftime.DatetimeNoLeap)
    for k in ("calendar", "units"):
        da[time_name].attrs.pop(k, None)
    print("time type:", type(da[time_name].values[0]))
    print("time first/last:", da[time_name].values[0], "...", da[time_name].values[-1])

    analyzer = QboAnalyzerFromScratch(
        QboConfig(
            u_var=varname,              # "ta"
            time_name=time_name,
            p_name=p_name,
            lat_name=lat_name,
            lon_name=lon_name,
            lat_band=lat_band,
            p_units_target=p_units_target,
            amp_remove_seasonal_cycle=anom,   # <--- IMPORTANT for temperature
        )
    )
    
    # Basic dimension checks
    need = {time_name, p_name}
    missing = need - set(da.dims)
    if missing:
        raise ValueError(f"{ref_dict['name']}: missing required dims {missing}. got dims={da.dims}")

    for d in (lat_name, lon_name):
        if d not in da.dims:
            print(f"WARNING: {ref_dict['name']}: '{d}' not in dims; skipping that averaging step.")

    # Sort time and drop duplicates
    da = da.sortby(time_name)
    tvals = da[time_name].values
    if len(tvals) != len(np.unique(tvals)):
        _, unique_idx = np.unique(tvals, return_index=True)
        da = da.isel({time_name: np.sort(unique_idx)})
        print(f"NOTE: dropped duplicate {time_name}; new ntime={da.sizes[time_name]}")

    # Ensure pressure coord has units (helps Pa->hPa conversion)
    if p_name in da.coords:
        if not (da[p_name].attrs.get("units") or "").strip():
            pmax = float(np.nanmax(da[p_name].values))
            da[p_name].attrs["units"] = "Pa" if pmax > 2000 else "hPa"

    p_units = da[p_name].attrs.get("units", "unknown") if p_name in da.coords else "no-coord"
    print(
        f"{vtag}: dims={da.dims} "
        f"shape={tuple(da.sizes[d] for d in da.dims)} "
        f"U.units={da.attrs.get('units', 'unknown')} p.units={p_units}"
    )

    # --- Make ERA5 vertical levels consistent with test model diagnostics ---
    target_plev_pa = np.array(plev, dtype=np.float64)  # your model levels in Pa

    # Convert lev coord to Pa if needed (based on metadata; fallback to magnitude)
    lev_units = (da[p_name].attrs.get("units") or "").strip().lower() if p_name in da.coords else ""
    print(lev_units)
    if lev_units in ("hpa", "millibar", "millibars", "mbar", "mb"):
        da = da.assign_coords({p_name: da[p_name] * 100.0})
        da[p_name].attrs["units"] = "Pa"
    elif lev_units in ("pa", "pascal", "pascals", ""):
        # If units missing, infer from typical magnitude
        if lev_units == "":
            pmax = float(np.nanmax(da[p_name].values))
            if pmax < 2000:  # likely hPa
                da = da.assign_coords({p_name: da[p_name] * 100.0})
                da[p_name].attrs["units"] = "Pa"
            else:
                da[p_name].attrs["units"] = "Pa"
    else:
        raise ValueError(f"Unrecognized {p_name} units: '{da[p_name].attrs.get('units')}'")
    
    # Select the target pressure levels
    # Use exact match first (common if levels are standard), otherwise nearest with tolerance.
    lev_vals = da[p_name].values.astype(np.float64)

    # Check which targets are exactly present
    exact = np.isin(target_plev_pa, lev_vals)
    if exact.all():
        da = da.sel({p_name: target_plev_pa})
    else:
        # nearest selection with a tolerance (Pa)
        # adjust tol if needed; 1 Pa is strict; 10-50 Pa is still very safe
        tol = 1.0
        da = da.sel({p_name: xr.DataArray(target_plev_pa, dims="plev_target")},
                    method="nearest", tolerance=tol)
    
        # Rename the temporary dimension back to lev, and overwrite coord with exact targets
        da = da.rename({"plev_target": p_name}).assign_coords({p_name: target_plev_pa})
    
        # sanity check: did we successfully match all requested levels?
        if da.sizes[p_name] != len(target_plev_pa):
            raise ValueError(
                f"After nearest selection, got {da.sizes[p_name]} levels but expected {len(target_plev_pa)}. "
                f"Try increasing tolerance."
            )
    
    # Optional: enforce same ordering as your model list (as provided)
    da = da.sel({p_name: target_plev_pa})

    # keep as 'lev' for analyzer
    metrics = analyzer.compute_metrics(
        name=f"{ref_dict['name']}_{varname}_{ref_dict['period']}",
        data=da,
        enable_spectral=False,
    )

    out_nc = out_dir / f"qbo_{ref_dict['name']}_{vtag}_{ref_dict['period']}.nc"
    analyzer.save_metrics(metrics, out_nc)
    print(f"Wrote {out_nc}")



=== ERA5 (2012-2013) ===
ta: nfiles=2 -> ['T_201201_201212.nc', 'T_201301_201312.nc']
Updated time coordinate:
time type: <class 'cftime._cftime.DatetimeNoLeap'>
time first/last: 2012-01-16 00:00:00 ... 2013-12-16 00:00:00
ta: dims=('time', 'plev', 'lat', 'lon') shape=(24, 37, 180, 360) U.units=K p.units=millibars
millibars


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/qbo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


Wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/qbo/qbo_ERA5_ta_2012-2013.nc


In [27]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "qbo"
    out_dir.mkdir(parents=True, exist_ok=True)

    ref_dict = {
        "name": "ERAI",
        "path": "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/ERAI",
        "period": "2012-2013",
    }
    
    plev = [ 100000, 92500, 85000, 70000, 60000, 50000, 40000, 30000, 25000, 20000, 15000, 10000, 7000, 5000, 3000, 2000, 1000, 500, 100] 
    
    var_cfg = {
        "tag": "ta",
        "sub": "post/atm/180x360_aave/ts/monthly/1yr",
        "pattern": "ta_*.nc",                 # matches U_201201_201212.nc, U_201301_201312.nc
        "name": "ta",
        "dims": ("time", "plev", "lat", "lon"),
        "lat_band": (-5.0, 5.0),
        "p_units_target": "hPa",
        "amp_remove_seasonal_cycle": True
    }

    vtag = var_cfg["tag"]
    varname = var_cfg["name"]
    anom = var_cfg["amp_remove_seasonal_cycle"]
    lat_band = var_cfg["lat_band"]
    p_units_target = var_cfg["p_units_target"]
    time_name, p_name, lat_name, lon_name = var_cfg["dims"]
    
    base = Path(ref_dict["path"]) / var_cfg["sub"]
    files = sorted(glob.glob(str(base / var_cfg["pattern"])))
    if not files:
        raise FileNotFoundError(f"No ERA5 files matched: {base / var_cfg['pattern']}")

    print(f"\n=== {ref_dict['name']} ({ref_dict['period']}) ===")
    print(f"{vtag}: nfiles={len(files)} -> {[Path(f).name for f in files]}")

    da = open_var_from_files(files, varname)
    if p_name == "lev":
        da = da.rename({"lev": "plev"})
        p_name = "plev"
        
    # --- Simple, consistent monthly time: set all months to YYYY-MM-16 00:00 ---
    print("Updated time coordinate:")
    da = set_monthly_time_to_midmonth_cftime(da, time_name=time_name, calendar_cls=cftime.DatetimeNoLeap)
    for k in ("calendar", "units"):
        da[time_name].attrs.pop(k, None)
    print("time type:", type(da[time_name].values[0]))
    print("time first/last:", da[time_name].values[0], "...", da[time_name].values[-1])

    analyzer = QboAnalyzerFromScratch(
        QboConfig(
            u_var=varname,              # "ta"
            time_name=time_name,
            p_name=p_name,
            lat_name=lat_name,
            lon_name=lon_name,
            lat_band=lat_band,
            p_units_target=p_units_target,
            amp_remove_seasonal_cycle=anom,   # <--- IMPORTANT for temperature
        )
    )
    
    # Basic dimension checks
    need = {time_name, p_name}
    missing = need - set(da.dims)
    if missing:
        raise ValueError(f"{ref_dict['name']}: missing required dims {missing}. got dims={da.dims}")

    for d in (lat_name, lon_name):
        if d not in da.dims:
            print(f"WARNING: {ref_dict['name']}: '{d}' not in dims; skipping that averaging step.")

    # Sort time and drop duplicates
    da = da.sortby(time_name)
    tvals = da[time_name].values
    if len(tvals) != len(np.unique(tvals)):
        _, unique_idx = np.unique(tvals, return_index=True)
        da = da.isel({time_name: np.sort(unique_idx)})
        print(f"NOTE: dropped duplicate {time_name}; new ntime={da.sizes[time_name]}")

    # Ensure pressure coord has units (helps Pa->hPa conversion)
    if p_name in da.coords:
        if not (da[p_name].attrs.get("units") or "").strip():
            pmax = float(np.nanmax(da[p_name].values))
            da[p_name].attrs["units"] = "Pa" if pmax > 2000 else "hPa"

    p_units = da[p_name].attrs.get("units", "unknown") if p_name in da.coords else "no-coord"
    print(
        f"{vtag}: dims={da.dims} "
        f"shape={tuple(da.sizes[d] for d in da.dims)} "
        f"U.units={da.attrs.get('units', 'unknown')} p.units={p_units}"
    )

    # --- Make ERA5 vertical levels consistent with test model diagnostics ---
    target_plev_pa = np.array(plev, dtype=np.float64)  # your model levels in Pa

    # Convert lev coord to Pa if needed (based on metadata; fallback to magnitude)
    lev_units = (da[p_name].attrs.get("units") or "").strip().lower() if p_name in da.coords else ""
    print(lev_units)
    if lev_units in ("hpa", "millibar", "millibars", "mbar", "mb"):
        da = da.assign_coords({p_name: da[p_name] * 100.0})
        da[p_name].attrs["units"] = "Pa"
    elif lev_units in ("pa", "pascal", "pascals", ""):
        # If units missing, infer from typical magnitude
        if lev_units == "":
            pmax = float(np.nanmax(da[p_name].values))
            if pmax < 2000:  # likely hPa
                da = da.assign_coords({p_name: da[p_name] * 100.0})
                da[p_name].attrs["units"] = "Pa"
            else:
                da[p_name].attrs["units"] = "Pa"
    else:
        raise ValueError(f"Unrecognized {p_name} units: '{da[p_name].attrs.get('units')}'")
    
    # Select the target pressure levels
    # Use exact match first (common if levels are standard), otherwise nearest with tolerance.
    lev_vals = da[p_name].values.astype(np.float64)

    # Check which targets are exactly present
    exact = np.isin(target_plev_pa, lev_vals)
    if exact.all():
        da = da.sel({p_name: target_plev_pa})
    else:
        # nearest selection with a tolerance (Pa)
        # adjust tol if needed; 1 Pa is strict; 10-50 Pa is still very safe
        tol = 1.0
        da = da.sel({p_name: xr.DataArray(target_plev_pa, dims="plev_target")},
                    method="nearest", tolerance=tol)
    
        # Rename the temporary dimension back to lev, and overwrite coord with exact targets
        da = da.rename({"plev_target": p_name}).assign_coords({p_name: target_plev_pa})
    
        # sanity check: did we successfully match all requested levels?
        if da.sizes[p_name] != len(target_plev_pa):
            raise ValueError(
                f"After nearest selection, got {da.sizes[p_name]} levels but expected {len(target_plev_pa)}. "
                f"Try increasing tolerance."
            )
    
    # Optional: enforce same ordering as your model list (as provided)
    da = da.sel({p_name: target_plev_pa})

    # keep as 'lev' for analyzer
    metrics = analyzer.compute_metrics(
        name=f"{ref_dict['name']}_{varname}_{ref_dict['period']}",
        data=da,
        enable_spectral=False,
    )

    out_nc = out_dir / f"qbo_{ref_dict['name']}_{vtag}_{ref_dict['period']}.nc"
    analyzer.save_metrics(metrics, out_nc)
    print(f"Wrote {out_nc}")



=== ERAI (2012-2013) ===
ta: nfiles=2 -> ['ta_201201_201212.nc', 'ta_201301_201312.nc']


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/qbo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


Updated time coordinate:
time type: <class 'cftime._cftime.DatetimeNoLeap'>
time first/last: 2012-01-16 00:00:00 ... 2013-12-16 00:00:00
ta: dims=('time', 'plev', 'lat', 'lon') shape=(24, 37, 241, 480) U.units=K p.units=Pa
pa
Wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/qbo/qbo_ERAI_ta_2012-2013.nc
